# Этап 4 — подготовка данных для Google Sheets

**Роль этого ноутбука:** подготовить и проверить clean-данные для новой Google Таблицы.

Python здесь не является source of truth для unit economics. Он делает только:
- проверку очищенных `data_clean/deals_clean.csv` и `data_clean/spend_clean.csv`;
- экспорт `data_clean/*.parquet` для Power BI;
- копирование полных clean CSV в `exports/google_sheets/`;
- опциональную загрузку этих двух CSV в raw-листы Google Sheets.

Финальные расчёты unit economics, дерево метрик, точки роста и гипотезы считаются в Google Sheets.

Новая таблица: https://docs.google.com/spreadsheets/d/1ybv6r_ZyLiyFJv0SggjvJQaiYa568UYCv9r6ml6uIsg

<a id="plan"></a>

## План

1. [Настройки](#settings)
2. [Экспорт Parquet](#parquet-export)
3. [Экспорт CSV для Google Sheets](#csv-export)
4. [Контроль качества выгрузки](#validation)
5. [Опциональная загрузка в Google Sheets](#sheets-upload)
6. [Что считается в Google Sheets](#sheets-formulas)

<a id="settings"></a>

## 1. Настройки

[Вернуться к плану](#plan)

In [ ]:
from pathlib import Path
import sys

import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

DATA_DIR = BASE_DIR / 'data_clean'
EXPORT_DIR = BASE_DIR / 'exports' / 'google_sheets'
SHEET_ID = '1ybv6r_ZyLiyFJv0SggjvJQaiYa568UYCv9r6ml6uIsg'
SHEET_URL = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}'

print(f'Project root: {BASE_DIR}')
print(f'Google Sheet: {SHEET_URL}')

<a id="parquet-export"></a>

## 2. Экспорт Parquet

[Вернуться к плану](#plan)

In [ ]:
from scripts.export_clean_parquet import export_clean_parquets

parquet_files = export_clean_parquets()
for file in parquet_files:
    print(file.relative_to(BASE_DIR))

<a id="csv-export"></a>

## 3. Экспорт CSV для Google Sheets

В Google Sheets грузятся не урезанные датасеты, а полные очищенные CSV: `deals_clean.csv` и `spend_clean.csv`. Ограничиваем только количество таблиц, потому что для базовой unit economics не нужны Calls и Contacts.

[Вернуться к плану](#plan)

In [ ]:
from scripts.export_google_sheets_inputs import export_google_sheets_inputs

csv_files = export_google_sheets_inputs()
for file in csv_files:
    print(file.relative_to(BASE_DIR))

<a id="validation"></a>

## 4. Контроль качества выгрузки

Проверяем, что Google Sheets получает те же самые clean CSV, без потери колонок и строк.

[Вернуться к плану](#plan)

In [ ]:
required_deals_columns = {
    'Id', 'Stage', 'Source', 'Campaign', 'Product', 'Payment Type',
    'Created Time', 'Closing Date', 'Initial Amount Paid', 'Offer Total Amount',
}
required_spend_columns = {'Date', 'Source', 'Campaign', 'Spend'}

checks = []
for filename, required_columns in {
    'deals_clean.csv': required_deals_columns,
    'spend_clean.csv': required_spend_columns,
}.items():
    clean = pd.read_csv(DATA_DIR / filename, low_memory=False)
    exported = pd.read_csv(EXPORT_DIR / filename, low_memory=False)

    missing = sorted(required_columns - set(exported.columns))
    same_columns = list(clean.columns) == list(exported.columns)
    same_rows = len(clean) == len(exported)

    checks.append({
        'file': filename,
        'rows': len(exported),
        'columns': len(exported.columns),
        'same_columns_as_clean': same_columns,
        'same_rows_as_clean': same_rows,
        'missing_required_columns': ', '.join(missing),
    })

validation = pd.DataFrame(checks)
display(validation)

assert validation['same_columns_as_clean'].all()
assert validation['same_rows_as_clean'].all()
assert (validation['missing_required_columns'] == '').all()

In [ ]:
deals = pd.read_csv(EXPORT_DIR / 'deals_clean.csv', low_memory=False)
spend = pd.read_csv(EXPORT_DIR / 'spend_clean.csv', low_memory=False)

deals['Initial Amount Paid'] = pd.to_numeric(deals['Initial Amount Paid'], errors='coerce')
spend['Spend'] = pd.to_numeric(spend['Spend'], errors='coerce')

paid_mask = (deals['Stage'] == 'Payment Done') & deals['Initial Amount Paid'].notna()
control = pd.DataFrame([
    ['Deals rows', len(deals)],
    ['Spend rows', len(spend)],
    ['Paid deals', int(paid_mask.sum())],
    ['First Payment Amount total', round(deals.loc[paid_mask, 'Initial Amount Paid'].sum(), 2)],
    ['Spend total', round(spend['Spend'].sum(), 2)],
], columns=['Control metric', 'Value'])

display(control)

<a id="sheets-upload"></a>

## 5. Опциональная загрузка в Google Sheets

Эта ячейка обновляет только raw-листы новой таблицы:
- `01_deals_clean_raw`;
- `02_spend_clean_raw`;
- `03_methodology`.

Если `google_credentials.json` не лежит в корне проекта и переменная `GOOGLE_APPLICATION_CREDENTIALS` не задана, ячейка ничего не загружает и просто показывает, что нужно добавить.

[Вернуться к плану](#plan)

In [ ]:
import os

from scripts.upload_google_sheets_clean_inputs import upload_clean_inputs_to_google_sheets

credentials_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS') or str(BASE_DIR / 'google_credentials.json')
if Path(credentials_path).exists():
    uploaded_url = upload_clean_inputs_to_google_sheets(sheet_id=SHEET_ID)
    print(f'Raw-листы обновлены: {uploaded_url}')
else:
    print('Google Sheets upload skipped.')
    print(f'Не найден credentials file: {credentials_path}')
    print('Добавь google_credentials.json в корень проекта или задай GOOGLE_APPLICATION_CREDENTIALS.')
    print(f'После этого запусти эту ячейку повторно для таблицы: {SHEET_URL}')

<a id="sheets-formulas"></a>

## 6. Что считается в Google Sheets

В Google Sheets считаем финальную unit economics по raw-листам:

- Source / Campaign / Product;
- Leads;
- Paid Deals;
- First Payment Amount;
- Spend;
- C1;
- CAC;
- CPL;
- Avg First Payment;
- First Payment Return / Spend;
- дерево метрик;
- точки роста и гипотезы.

`Initial Amount Paid` в базовой аналитике называем **First Payment Amount**, а не полной выручкой. Сценарный блок revenue recognition делается отдельно в `03_analysis.ipynb`, чтобы не смешивать первый платёж с признанной выручкой.

[Вернуться к плану](#plan)